In [1]:
# Install dependencies (add imbalanced-learn)
%pip install pandas matplotlib scikit-learn imbalanced-learn keras tensorflow --quiet

from features import get_train_test_data

data = get_train_test_data("data/bank_data_train.csv", sampler="none")

print(f"x_train shape: {data.x_train.shape}")
print(f"x_test shape: {data.x_test.shape}")
print(f"y_train shape: {data.y_train.shape}")
print(f"y_test shape: {data.y_test.shape}")

# Display first 5 rows of x_train
data.x_train.head()

/home/samy/churn/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
x_train shape: (284152, 44)
x_test shape: (71038, 44)
y_train shape: (284152,)
y_test shape: (71038,)


,CR_PROD_CNT_IL,TURNOVER_DYNAMIC_IL_1M,REST_DYNAMIC_FDEP_1M,REST_DYNAMIC_SAVE_3M,CR_PROD_CNT_VCU,REST_AVG_CUR,CR_PROD_CNT_TOVR,CR_PROD_CNT_PIL,TURNOVER_CC,TURNOVER_PAYM,...,PACK_103,PACK_104,PACK_105,PACK_107,PACK_108,PACK_109,PACK_301,PACK_K01,PACK_M01,PACK_O01
54105,2.061893,-0.044935,-0.052273,-0.312231,5.244695,-0.110851,1.176112,-0.192559,-0.038441,-0.093385,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
313302,-0.244365,-0.044935,-0.052273,2.151389,-0.169621,0.192015,-0.526426,-0.192559,-0.038441,-0.093385,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
302648,-0.244365,-0.044935,-0.052273,-0.312231,-0.169621,-0.260938,-0.526426,-0.192559,-0.038441,-0.093385,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
350887,2.061893,-0.044935,-0.052273,-0.312231,-0.169621,-0.332548,1.176112,-0.192559,-0.038441,-0.093385,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41493,-0.244365,-0.044935,-0.052273,-0.312231,-0.169621,-0.291020,1.176112,-0.192559,-0.038441,-0.093385,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [2]:
# print name of features used for training
print(f"Features used for training: {data.x_train.columns.tolist()}")

Features used for training: ['CR_PROD_CNT_IL', 'TURNOVER_DYNAMIC_IL_1M', 'REST_DYNAMIC_FDEP_1M', 'REST_DYNAMIC_SAVE_3M', 'CR_PROD_CNT_VCU', 'REST_AVG_CUR', 'CR_PROD_CNT_TOVR', 'CR_PROD_CNT_PIL', 'TURNOVER_CC', 'TURNOVER_PAYM', 'AGE', 'CR_PROD_CNT_CC', 'REST_DYNAMIC_FDEP_3M', 'REST_DYNAMIC_IL_1M', 'CR_PROD_CNT_CCFP', 'REST_DYNAMIC_CUR_1M', 'REST_AVG_PAYM', 'LDEAL_GRACE_DAYS_PCT_MED', 'REST_DYNAMIC_CUR_3M', 'TURNOVER_DYNAMIC_CUR_1M', 'REST_DYNAMIC_PAYM_3M', 'REST_DYNAMIC_IL_3M', 'TURNOVER_DYNAMIC_IL_3M', 'REST_DYNAMIC_PAYM_1M', 'TURNOVER_DYNAMIC_CUR_3M', 'CLNT_SETUP_TENOR', 'TURNOVER_DYNAMIC_PAYM_3M', 'TURNOVER_DYNAMIC_PAYM_1M', 'REST_DYNAMIC_CC_1M', 'TURNOVER_DYNAMIC_CC_1M', 'REST_DYNAMIC_CC_3M', 'TURNOVER_DYNAMIC_CC_3M', 'PACK_101', 'PACK_102', 'PACK_103', 'PACK_104', 'PACK_105', 'PACK_107', 'PACK_108', 'PACK_109', 'PACK_301', 'PACK_K01', 'PACK_M01', 'PACK_O01']


In [3]:
import tensorflow as tf

LIBRARYNAME = "TensorFlow"
ALGORITHMNAME = "Feedforward Neural Network"
LOSS_FUNCTION = "binary_crossentropy"
LEARNING_RATE = 0.001
EPOCHS = 60
BATCH_SIZE = 1024

from tensorflow.keras import regularizers

def create_model(input_dim, dropout_rate=0.1, l2_reg=1e-9):
    inputs = tf.keras.Input(shape=(input_dim,))

    # Hidden layer 1
    x = tf.keras.layers.Dense(512, activation='relu',
                               kernel_regularizer=regularizers.l2(l2_reg))(inputs)
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    # Hidden layer 2
    x = tf.keras.layers.Dense(512, activation='relu',
                               kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    # Output layer
    outputs = tf.keras.layers.Dense(1, activation='sigmoid',
                                    kernel_regularizer=regularizers.l2(l2_reg))(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs)

model = create_model(data.x_train.shape[1])

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=LOSS_FUNCTION,
    metrics=[tf.keras.metrics.AUC(name='auc')]
)

# Custom callback to stop at target AUC
class StopAtTargetAUC(tf.keras.callbacks.Callback):
    def __init__(self, target_auc=0.85):
        super().__init__()
        self.target_auc = target_auc
    
    def on_epoch_end(self, _, logs=None):
        val_auc = logs.get('auc')
        if val_auc is not None:
            val_auc = float(val_auc)
            if val_auc >= self.target_auc:
                print(f"\nReached target AUC of {self.target_auc}! Stopping training.")
                self.model.stop_training = True

# Train model
history = model.fit(
    data.x_train,
    data.y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[StopAtTargetAUC(target_auc=0.85)],
    verbose=1
)

I0000 00:00:1778992359.554915   86427 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778992359.625732   86427 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778992361.716274   86427 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1778992362.338273   86427 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries men

Epoch 1/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - auc: 0.6940 - loss: 0.2695
Epoch 2/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - auc: 0.7641 - loss: 0.2471
Epoch 3/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step - auc: 0.7814 - loss: 0.2418
Epoch 4/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - auc: 0.7901 - loss: 0.2391
Epoch 5/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - auc: 0.7972 - loss: 0.2366
Epoch 6/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - auc: 0.8030 - loss: 0.2347
Epoch 7/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - auc: 0.8075 - loss: 0.2330
Epoch 8/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - auc: 0.8104 - loss: 0.2320
Epoch 9/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - auc: 0.8122 - loss: 0.2312
Epoch 10/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - auc: 0.8151 - loss: 0.2302
Epoch 11/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - auc: 0.8179 - loss: 0.2290
Epoch 12/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - auc: 0.8184 - loss: 0.2287
Epoch 13/60


In [4]:
import pandas as pd

metrics = model.get_metrics_result()

LOSS = metrics["loss"]
AUC = metrics["auc"]

print(model.summary())

pd.DataFrame({
    'Library': [LIBRARYNAME],
    'Algorithm': [ALGORITHMNAME],
    'LossFunction': [LOSS_FUNCTION],
    'LearningRate': [LEARNING_RATE],
    'Epochs': [EPOCHS],
    'BatchSize': [BATCH_SIZE],
    'AUC': [AUC],
    'Loss': [LOSS]
}).T

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 44)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │        23,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 858,629 (3.28 MB)

 Trainable params: 286,209 (1.09 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 572,420 (2.18 MB)

None


,0
Library,TensorFlow
Algorithm,Feedforward Neural Network
LossFunction,binary_crossentropy
LearningRate,0.001
Epochs,60
BatchSize,1024
AUC,"tf.Tensor(0.8509471, shape=(), dtype=float32)"
Loss,"tf.Tensor(0.21298796, shape=(), dtype=float32)"


In [6]:
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import regularizers
from features import get_train_test_data

# ── Defaults ──────────────────────────────────────────────────────────────────
DEFAULTS = {
    "sampler":       "none",
    "hidden_units":  512,
    "dropout_rate":  0.1,
    "l2_reg":        1e-9,
    "learning_rate": 0.001,
    "batch_size":    1024,
}

# this is a square matrix (for sake of simplicity and speed of evaluation)
# ── Values to try per param ───────────────────────────────────────────────────

param_grid = [
    # "sampler":       ["none", "smote", "oversample", "undersample"],
    # "hidden_units":  [64, 128, 256, 512],
    # "dropout_rate":  [0.1, 0.3, 0.5],
    # "l2_reg":        [1e-4, 1e-5, 0.0],
    # "learning_rate": [0.001, 0.0005],
    # "batch_size":    [512, 1024],
("", {
    "sampler": "none",
    "hidden_units": 512,
    "dropout_rate": 0.1,
    "l2_reg": 1e-9,
    "learning_rate": 0.001,
    "batch_size": 1024
})
]

EPOCHS    = 30
PATIENCE  = 8
DATA_PATH = "data/bank_data_train.csv"

# ── Model factory ─────────────────────────────────────────────────────────────
def build_model(input_dim, hidden_units, dropout_rate, l2_reg, learning_rate):
    tf.keras.backend.clear_session()
    reg    = regularizers.l2(l2_reg) if l2_reg > 0 else None
    inputs = tf.keras.Input(shape=(input_dim,))

    x = tf.keras.layers.Dense(hidden_units, activation="relu",
                               kernel_regularizer=reg)(inputs)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    x = tf.keras.layers.Dense(hidden_units, activation="relu",
                               kernel_regularizer=reg)(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid",
                                    kernel_regularizer=reg)(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")],
    )
    return model

# ── One-at-a-time search ──────────────────────────────────────────────────────
results     = []
best_auc    = -1
# best_params = DEFAULTS.copy()

for param_name, params in param_grid:
    print(f"\n{'='*60}")
    print(f"  Searching: {param_name}  (others fixed at defaults)")
    print(f"{'='*60}")

    # for value in values:
        # Build params with one value swapped
        # params = {**best_params, param_name: value}
        # print(f"\n  {param_name} = {value}  |  {params}")

    data = get_train_test_data(
        DATA_PATH,
        sampler         = params["sampler"],
        validation_size = 0.1,
    )

    m = build_model(
        input_dim    = data.x_train.shape[1],
        hidden_units = params["hidden_units"],
        dropout_rate = params["dropout_rate"],
        l2_reg       = params["l2_reg"],
        learning_rate= params["learning_rate"],
    )

    history = m.fit(
        data.x_train, data.y_train,
        epochs          = EPOCHS,
        batch_size      = params["batch_size"],
        validation_data = (data.x_val, data.y_val),
        callbacks       = [
            tf.keras.callbacks.EarlyStopping(
                # monitor             = "auc",
                monitor             = "auc",
                # patience            = PATIENCE,
                # restore_best_weights= True,
                # mode                = "max",
                # verbose             = 0,
            )
        ],
        verbose = 0,
    )

    auc = max(history.history["auc"])
    print(f"  → auc = {auc:.4f}")

    results.append({**params, "varied_param": param_name, "auc": auc})

    if auc > best_auc:
        best_auc               = auc
        best_params            = params.copy()  # update best so next params build on it
        # best_params[param_name] = value

    del m, history, data
    tf.keras.backend.clear_session()
    gc.collect()

# ── Results table ─────────────────────────────────────────────────────────────
results_df = (
    pd.DataFrame(results)
    .sort_values("auc", ascending=False)
    .reset_index(drop=True)
)
results_df.index += 1

print("\n── Ranked results ───────────────────────────────────────────")
print(results_df.to_string())

print(f"\n── Best AUC : {best_auc:.4f} ────────────────────────────")
print("── Best params ──────────────────────────────────────────────")
for k, v in best_params.items():
    print(f"  {k:<20} {v}")


  Searching:   (others fixed at defaults)
  → auc = 0.8415

── Ranked results ───────────────────────────────────────────
  sampler  hidden_units  dropout_rate        l2_reg  learning_rate  batch_size varied_param      auc
1    none           512           0.1  1.000000e-09          0.001        1024               0.84149

── Best AUC : 0.8415 ────────────────────────────
── Best params ──────────────────────────────────────────────
  sampler              none
  hidden_units         512
  dropout_rate         0.1
  l2_reg               1e-09
  learning_rate        0.001
  batch_size           1024
